# ✅ Boosting Methods — Solutions Notebook
### Complete solutions for Notebook 01 (Concepts) + Notebook 02 (Applied) Capstone Exercises

---

**Is file mein dono notebooks ke capstone exercises ke complete solutions hain.**
- **Part 1:** Notebook 01 — Heart Disease Capstone (TODO A–D)
- **Part 2:** Notebook 02 — Diabetes Prediction Capstone (TODO A–H)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

print("✅ Ready!")

---
## 🏆 PART 1: Notebook 01 — Loan Default Capstone Solutions

**Original Capstone (from 01_Boosting_Concepts):** Loan Default dataset pe AdaBoost aur GBM apply karna

In [ ]:
# --- Reconstruct the Concepts Notebook Dataset ---
np.random.seed(42)
loan_data = pd.DataFrame({
    'Age': [25, 45, 35, 50, 23, 40, 60, 30, 55, 42,
             28, 38, 52, 47, 33],
    'Income_L': [2.5, 8.0, 5.5, 12.0, 1.8, 7.0, 15.0, 3.2, 10.5, 6.8,
                  2.1, 5.0, 11.0, 9.5, 4.3],
    'Debt_Ratio': [0.35, 0.15, 0.25, 0.10, 0.60, 0.20, 0.08, 0.45, 0.12, 0.22,
                    0.55, 0.30, 0.09, 0.18, 0.40],
    'Credit_Score': [620, 780, 710, 820, 580, 760, 850, 650, 800, 730,
                      600, 695, 815, 775, 660],
    'Default': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0,
                 1, 0, 0, 0, 1]
})

print("Loan Dataset loaded:")
print(loan_data.head())
print("\nDefault distribution:")
print(loan_data['Default'].value_counts())

In [ ]:
# ============================================================
# SOLUTION — TODO A: Features aur Target separate karo
# ============================================================
X_loan = loan_data.drop('Default', axis=1)
y_loan = loan_data['Default']

print("X shape:", X_loan.shape)
print("y shape:", y_loan.shape)
print("Features:", list(X_loan.columns))

In [ ]:
# ============================================================
# SOLUTION — TODO B: Train-test split (small dataset → 70/30)
# ============================================================
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_loan, y_loan, test_size=0.3, random_state=42
    # Note: stratify=y nahi dala kyunki dataset bahut chota hai (15 rows)
)

print("Train size:", X_train_l.shape)
print("Test size:", X_test_l.shape)

In [ ]:
# ============================================================
# SOLUTION — TODO C: AdaBoost train + evaluate
# ============================================================
ada_loan = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=20,   # small dataset → fewer estimators
    algorithm='SAMME',
    random_state=42
)
ada_loan.fit(X_train_l, y_train_l)

ada_loan_pred = ada_loan.predict(X_test_l)
ada_loan_acc = accuracy_score(y_test_l, ada_loan_pred)

print("=" * 35)
print("AdaBoost — Loan Default")
print("=" * 35)
print(f"Test Accuracy: {ada_loan_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_l, ada_loan_pred,
                             target_names=['No Default', 'Default']))

In [ ]:
# ============================================================
# SOLUTION — TODO D: GBM train + evaluate + compare
# ============================================================
gbm_loan = GradientBoostingClassifier(
    n_estimators=20,
    learning_rate=0.2,
    max_depth=2,
    random_state=42
)
gbm_loan.fit(X_train_l, y_train_l)

gbm_loan_pred = gbm_loan.predict(X_test_l)
gbm_loan_acc = accuracy_score(y_test_l, gbm_loan_pred)

print("=" * 35)
print("Gradient Boosting — Loan Default")
print("=" * 35)
print(f"Test Accuracy: {gbm_loan_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_l, gbm_loan_pred,
                             target_names=['No Default', 'Default']))

In [ ]:
# --- Comparison Plot ---
dt_loan = DecisionTreeClassifier(random_state=42)
dt_loan.fit(X_train_l, y_train_l)
dt_loan_acc = dt_loan.score(X_test_l, y_test_l)

models = ['Decision Tree', 'AdaBoost', 'GBM']
accs = [dt_loan_acc, ada_loan_acc, gbm_loan_acc]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bar chart
colors = ['#95a5a6', '#9b59b6', '#27ae60']
bars = axes[0].bar(models, accs, color=colors)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Accuracy Comparison')
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', fontweight='bold')

# AdaBoost confusion matrix
ConfusionMatrixDisplay.from_estimator(
    ada_loan, X_test_l, y_test_l,
    display_labels=['No Default', 'Default'],
    cmap='Purples', ax=axes[1]
)
axes[1].set_title('AdaBoost CM')

# GBM confusion matrix
ConfusionMatrixDisplay.from_estimator(
    gbm_loan, X_test_l, y_test_l,
    display_labels=['No Default', 'Default'],
    cmap='Greens', ax=axes[2]
)
axes[2].set_title('GBM CM')

plt.suptitle('Loan Default — Boosting Solutions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Importance (GBM) ---
feat_imp_loan = pd.Series(gbm_loan.feature_importances_, index=X_loan.columns).sort_values()

plt.figure(figsize=(6, 3))
feat_imp_loan.plot(kind='barh', color='#27ae60')
plt.title('GBM — Loan Default Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nInsight: Sabse important feature hai:", feat_imp_loan.idxmax())
print("Matlab: Bank ke liye", feat_imp_loan.idxmax(), "loan default predict karne mein sabse useful hai")

---
## 🏆 PART 2: Notebook 02 — Diabetes Capstone Solutions

**Original Capstone (from 02_Boosting_Applied):** Diabetes dataset pe TODO A–H complete karna

In [ ]:
# --- Recreate the same Diabetes dataset ---
np.random.seed(42)
n = 500
diabetes_df = pd.DataFrame({
    'Pregnancies': np.random.randint(0, 15, n),
    'Glucose': np.random.randint(70, 200, n),
    'BloodPressure': np.random.randint(40, 110, n),
    'SkinThickness': np.random.randint(10, 60, n),
    'Insulin': np.random.randint(0, 300, n),
    'BMI': np.round(np.random.uniform(18, 50, n), 1),
    'DiabetesPedigreeFunction': np.round(np.random.uniform(0.08, 2.5, n), 3),
    'Age': np.random.randint(21, 81, n)
})
prob = (diabetes_df['Glucose']/200 * 0.5 +
        diabetes_df['BMI']/50 * 0.3 +
        diabetes_df['Age']/80 * 0.2)
diabetes_df['Outcome'] = (prob + np.random.normal(0, 0.1, n) > 0.5).astype(int)

print("Diabetes Dataset loaded:")
print(diabetes_df.shape)
print(diabetes_df['Outcome'].value_counts())

In [ ]:
# ============================================================
# SOLUTION — TODO A: X aur y separate karo
# ============================================================
X_diab = diabetes_df.drop('Outcome', axis=1)
y_diab = diabetes_df['Outcome']

print("X shape:", X_diab.shape)
print("y shape:", y_diab.shape)
print("Features:", list(X_diab.columns))

In [ ]:
# ============================================================
# SOLUTION — TODO B: Train-test split (stratified)
# ============================================================
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=42, stratify=y_diab
)

print("Train:", X_train_d.shape, "| Test:", X_test_d.shape)
print("\nTrain class %:", y_train_d.value_counts(normalize=True).round(3).to_dict())
print("Test class %:", y_test_d.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# ============================================================
# SOLUTION — TODO C: StandardScaler
# ============================================================
scaler_d = StandardScaler()
X_train_ds = scaler_d.fit_transform(X_train_d)
X_test_ds = scaler_d.transform(X_test_d)

print("Scaling complete!")
print("Mean after scaling:", X_train_ds.mean().round(4))  # ~0
print("Std after scaling:", X_train_ds.std().round(4))    # ~1

In [ ]:
# ============================================================
# SOLUTION — TODO D: AdaBoost train karo
# ============================================================
ada_diab = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=100,
    algorithm='SAMME',
    random_state=42
)
ada_diab.fit(X_train_ds, y_train_d)

ada_d_train = ada_diab.score(X_train_ds, y_train_d)
ada_d_test = ada_diab.score(X_test_ds, y_test_d)
ada_d_cv = cross_val_score(ada_diab, X_train_ds, y_train_d, cv=5)

print(f"AdaBoost Train: {ada_d_train:.4f}")
print(f"AdaBoost Test : {ada_d_test:.4f}")
print(f"AdaBoost CV   : {ada_d_cv.mean():.4f} ± {ada_d_cv.std():.4f}")

In [ ]:
# ============================================================
# SOLUTION — TODO E: GBM train karo
# ============================================================
gbm_diab = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
gbm_diab.fit(X_train_ds, y_train_d)

gbm_d_train = gbm_diab.score(X_train_ds, y_train_d)
gbm_d_test = gbm_diab.score(X_test_ds, y_test_d)
gbm_d_cv = cross_val_score(gbm_diab, X_train_ds, y_train_d, cv=5)

print(f"GBM Train: {gbm_d_train:.4f}")
print(f"GBM Test : {gbm_d_test:.4f}")
print(f"GBM CV   : {gbm_d_cv.mean():.4f} ± {gbm_d_cv.std():.4f}")

In [ ]:
# ============================================================
# SOLUTION — TODO F: Accuracy compare karo
# ============================================================
dt_diab = DecisionTreeClassifier(random_state=42)
dt_diab.fit(X_train_ds, y_train_d)
dt_d_test = dt_diab.score(X_test_ds, y_test_d)

comp = pd.DataFrame({
    'Model': ['Decision Tree', 'AdaBoost', 'GBM'],
    'Test Acc': [dt_d_test, ada_d_test, gbm_d_test],
    'CV Mean': [
        cross_val_score(dt_diab, X_train_ds, y_train_d, cv=5).mean(),
        ada_d_cv.mean(),
        gbm_d_cv.mean()
    ]
})
comp[['Test Acc', 'CV Mean']] = comp[['Test Acc', 'CV Mean']].round(4)
print(comp.sort_values('Test Acc', ascending=False).to_string(index=False))

winner = comp.loc[comp['Test Acc'].idxmax(), 'Model']
print(f"\n🏆 Best Model: {winner}")

In [ ]:
# ============================================================
# SOLUTION — TODO G: Confusion Matrices side by side
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_estimator(
    ada_diab, X_test_ds, y_test_d,
    display_labels=['No Diabetes', 'Diabetes'],
    cmap='Purples', ax=axes[0]
)
axes[0].set_title(f'AdaBoost (Acc: {ada_d_test:.3f})')

ConfusionMatrixDisplay.from_estimator(
    gbm_diab, X_test_ds, y_test_d,
    display_labels=['No Diabetes', 'Diabetes'],
    cmap='Greens', ax=axes[1]
)
axes[1].set_title(f'GBM (Acc: {gbm_d_test:.3f})')

plt.suptitle('Diabetes Prediction — Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SOLUTION — TODO H: Feature Importance (GBM)
# ============================================================
gbm_feat_diab = pd.Series(gbm_diab.feature_importances_, index=X_diab.columns).sort_values()

plt.figure(figsize=(8, 4))
gbm_feat_diab.plot(kind='barh', color='#27ae60')
plt.xlabel('Feature Importance')
plt.title('GBM — Diabetes Feature Importance', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Feature Importance Ranking:")
print(gbm_feat_diab.sort_values(ascending=False).round(4))

print("\n💡 Insight:")
top3 = gbm_feat_diab.nlargest(3).index.tolist()
print(f"Top 3 features: {top3}")
print("Matlab: Diabetes predict karne mein yeh features sabse zyada important hain")

In [ ]:
# --- Detailed Classification Reports ---
print("=" * 50)
print("AdaBoost — Classification Report")
print("=" * 50)
ada_pred_d = ada_diab.predict(X_test_ds)
print(classification_report(y_test_d, ada_pred_d, target_names=['No Diabetes', 'Diabetes']))

print("=" * 50)
print("GBM — Classification Report")
print("=" * 50)
gbm_pred_d = gbm_diab.predict(X_test_ds)
print(classification_report(y_test_d, gbm_pred_d, target_names=['No Diabetes', 'Diabetes']))

In [ ]:
# --- Bonus: Predict for a New Patient ---
# Ek naye patient ke liye predict karna
new_patient = pd.DataFrame([{
    'Pregnancies': 3,
    'Glucose': 160,        # High glucose
    'BloodPressure': 85,
    'SkinThickness': 30,
    'Insulin': 100,
    'BMI': 32.5,           # Overweight
    'DiabetesPedigreeFunction': 0.6,
    'Age': 45
}])

new_patient_scaled = scaler_d.transform(new_patient)

ada_pred_new = ada_diab.predict(new_patient_scaled)[0]
gbm_pred_new = gbm_diab.predict(new_patient_scaled)[0]
ada_prob = ada_diab.predict_proba(new_patient_scaled)[0]
gbm_prob = gbm_diab.predict_proba(new_patient_scaled)[0]

print("=" * 40)
print("NEW PATIENT PREDICTION")
print("=" * 40)
print(f"AdaBoost Prediction: {'Diabetes ⚠️' if ada_pred_new == 1 else 'No Diabetes ✅'}")
print(f"AdaBoost Probability: No={ada_prob[0]:.2f}, Yes={ada_prob[1]:.2f}")
print()
print(f"GBM Prediction: {'Diabetes ⚠️' if gbm_pred_new == 1 else 'No Diabetes ✅'}")
print(f"GBM Probability: No={gbm_prob[0]:.2f}, Yes={gbm_prob[1]:.2f}")

---
## 🎓 Solutions Summary

### Part 1 — Loan Default Capstone:
- **TODO A:** `X = loan_data.drop('Default', axis=1)`, `y = loan_data['Default']`
- **TODO B:** `train_test_split(..., test_size=0.3)` — small dataset mein 70/30 better
- **TODO C:** `AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), algorithm='SAMME')`
- **TODO D:** `GradientBoostingClassifier(n_estimators=20, learning_rate=0.2, max_depth=2)`

### Part 2 — Diabetes Capstone:
- **TODO A:** Features aur target separate karna
- **TODO B:** `train_test_split(..., stratify=y)` — class proportion maintain karna
- **TODO C:** `StandardScaler().fit_transform()` — scaling
- **TODO D:** `AdaBoostClassifier(n_estimators=100, algorithm='SAMME')`
- **TODO E:** `GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, subsample=0.8)`
- **TODO F:** Comparison table with `.score()` and `cross_val_score()`
- **TODO G:** `ConfusionMatrixDisplay.from_estimator()` — side by side
- **TODO H:** `pd.Series(model.feature_importances_, index=X.columns).plot(kind='barh')`

---

### 🌍 Key Takeaways:
1. **Boosting always beats single tree** — kyunki wo errors pe iterate karta hai
2. **GBM > AdaBoost** generally — zyada control hai hyperparameters pe
3. **Feature Importance** batata hai ki kaunsa feature zyada predict karta hai — doctors ke liye useful!
4. **New patient prediction** — trained model ko kisi bhi naye data pe apply kar sakte hain